In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# 5-Class Random Forest Classifier with 5-Fold Cross-Validation (`models/rf_all.ipynb`)

This notebook evaluates a **5-Class Random Forest Classifier** for **ESI 1, 2, 3, 4, and 5** incorporating **29 Predictor Features** using **5-Fold Stratified Cross-Validation**:

### Predictor Feature Inventory (29 Predictor Features Total)
1. **Baseline Features (3)**: `age`, `gender`, `cc_breathingdifficulty`.
2. **10 Binary Vital Anomaly Flags**:
   - `is_dyspnea_total`: `triage_vital_o2 < 90`
   - `is_dyspnea_moderate`: `triage_vital_o2 >= 90 & triage_vital_o2 < 94`
   - `is_bradypnea`: `triage_vital_rr < 10`
   - `is_tachypnea`: `triage_vital_rr > 30`
   - `is_hypotension`: `triage_vital_sbp <= 90`
   - `is_hypertension`: `triage_vital_sbp > 220`
   - `is_bradycardia_total`: `triage_vital_hr < 40`
   - `is_bradycardia_moderate`: `triage_vital_hr >= 40 & triage_vital_hr < 60`
   - `is_tachycardia_total`: `triage_vital_hr > 150`
   - `is_tachycardia_moderate`: `triage_vital_hr > 100 & triage_vital_hr <= 150`
3. **16 Continuous Vital Delta & Range Features**:
   - `hr_mean_to_last`: `triage_vital_hr - pulse_last`
   - `sbp_mean_to_last`: `triage_vital_sbp - sbp_last`
   - `spo2_mean_to_last`: `triage_vital_o2 - spo2_last`
   - `rr_mean_to_last`: `triage_vital_rr - resp_last`
   - `hr_range`: `pulse_max - pulse_min`
   - `rr_range`: `resp_max - resp_min`
   - `spo2_range`: `spo2_max - spo2_min`
   - `sbp_range`: `sbp_max - sbp_min`
   - `hr_last_to_min`: `pulse_last - pulse_min`
   - `rr_last_to_min`: `resp_last - resp_min`
   - `spo2_last_to_min`: `spo2_last - spo2_min`
   - `sbp_last_to_min`: `sbp_last - sbp_min`
   - `hr_last_to_max`: `pulse_last - pulse_max`
   - `rr_last_to_max`: `resp_last - resp_max`
   - `spo2_last_to_max`: `spo2_last - spo2_max`
   - `sbp_last_to_max`: `sbp_last - sbp_max`

### 5-Fold Stratified Cross-Validation Protocol
- Splits full dataset into 5 equal stratified folds ($K=5$).
- Trains Random Forest on 4 folds (80%) and predicts out-of-fold probabilities on the remaining fold (20%).
- Reports per-fold and overall out-of-fold (OOF) **Accuracy, Precision, Recall, F1-Score, ROC-AUC, and Confusion Matrix**.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(caret)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pROC)
  if (requireNamespace("ranger", quietly = TRUE)) {
    library(ranger)
  } else {
    library(randomForest)
  }
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}
config <- fromJSON(config_path)
cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data & Construct 29 Predictor Features
# ---------------------------------------------------------
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}
cat("Loading dataset from:", data_file, "...\n")
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))
raw_df <- get(data_obj_name, envir = data_env)
target_col <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
p_last   <- get_vec("pulse_last")
p_max    <- get_vec("pulse_max")
p_min    <- get_vec("pulse_min")
s_last   <- get_vec("sbp_last")
s_max    <- get_vec("sbp_max")
s_min    <- get_vec("sbp_min")
o2_last  <- get_vec("spo2_last")
o2_max   <- get_vec("spo2_max")
o2_min   <- get_vec("spo2_min")
r_last   <- get_vec("resp_last")
r_max    <- get_vec("resp_max")
r_min    <- get_vec("resp_min")
t_hr     <- get_vec("triage_vital_hr")
t_sbp    <- get_vec("triage_vital_sbp")
t_o2     <- get_vec("triage_vital_o2")
t_rr     <- get_vec("triage_vital_rr")
# Construct 29 Features: 3 baseline + 10 binary anomaly flags + 16 vital delta & range features
df_full <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0),
  
  hr_mean_to_last         = t_hr - p_last,
  sbp_mean_to_last        = t_sbp - s_last,
  spo2_mean_to_last       = t_o2 - o2_last,
  rr_mean_to_last         = t_rr - r_last,
  
  hr_range                = p_max - p_min,
  rr_range                = r_max - r_min,
  spo2_range              = o2_max - o2_min,
  sbp_range               = s_max - s_min,
  
  hr_last_to_min          = p_last - p_min,
  rr_last_to_min          = r_last - r_min,
  spo2_last_to_min        = o2_last - o2_min,
  sbp_last_to_min         = s_last - s_min,
  
  hr_last_to_max          = p_last - p_max,
  rr_last_to_max          = r_last - r_max,
  spo2_last_to_max        = o2_last - o2_max,
  sbp_last_to_max         = s_last - s_max
)
raw_esi <- as.character(raw_df[[target_col]])
df_full$target_col <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
initial_rows <- nrow(df_full)
df_full <- na.omit(df_full)
cat(sprintf("Complete Case Filtering: Removed %d rows with NULL/NA features (Remaining complete rows: %d)\n",
            initial_rows - nrow(df_full), nrow(df_full)))
cat(sprintf("Full Dataset Ready for 5-Fold CV: %d total rows x %d cols\n", nrow(df_full), ncol(df_full)))
cat("Predictor Features Included (29 Total Features):\n")
print(setdiff(names(df_full), "target_col"))
cat("\nNatural 5-Class Target Distribution ('1', '2', '3', '4', '5'):\n")
print(table(df_full$target_col))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Perform 5-Fold Stratified Cross-Validation Loop
# ---------------------------------------------------------
set.seed(config$training$random_state)
k_folds <- 5
folds   <- createFolds(df_full$target_col, k = k_folds, list = TRUE, returnTrain = FALSE)
oof_prob <- matrix(0, nrow = nrow(df_full), ncol = 5)
colnames(oof_prob) <- c("1", "2", "3", "4", "5")
binary_cols <- c("gender", "cc_breathingdifficulty",
                 "is_dyspnea_total", "is_dyspnea_moderate", "is_bradypnea", "is_tachypnea",
                 "is_hypotension", "is_hypertension", "is_bradycardia_total", "is_bradycardia_moderate",
                 "is_tachycardia_total", "is_tachycardia_moderate")
cont_cols <- setdiff(names(df_full), c(binary_cols, "target_col"))
fold_metrics <- list()
cat("============================================================\n")
cat(sprintf("   STARTING %d-FOLD STRATIFIED CROSS-VALIDATION RUN\n", k_folds))
cat("============================================================\n")
is_ranger <- requireNamespace("ranger", quietly = TRUE)
for (k in 1:k_folds) {
  test_idx   <- folds[[k]]
  train_fold <- df_full[-test_idx, ]
  val_fold   <- df_full[test_idx, ]
  
  # Standardize continuous features per fold
  preproc_fold <- preProcess(train_fold[, cont_cols, drop = FALSE], method = c("center", "scale"))
  train_fold   <- predict(preproc_fold, train_fold)
  val_fold     <- predict(preproc_fold, val_fold)
  
  if (is_ranger) {
    rf_fold <- ranger::ranger(
      formula     = target_col ~ . ,
      data        = train_fold,
      num.trees   = 100,
      probability = TRUE,
      seed        = config$training$random_state + k,
      verbose     = FALSE
    )
    probs <- predict(rf_fold, data = val_fold)$predictions
  } else {
    rf_fold <- randomForest::randomForest(target_col ~ ., data = train_fold, ntree = 100)
    probs   <- predict(rf_fold, newdata = val_fold, type = "prob")
  }
  
  oof_prob[test_idx, ] <- probs
  
  # Calculate fold-level accuracy
  fold_pred_idx <- apply(probs, 1, which.max)
  fold_pred_fac <- factor(colnames(probs)[fold_pred_idx], levels = c("1", "2", "3", "4", "5"))
  fold_cm       <- confusionMatrix(fold_pred_fac, val_fold$target_col)
  fold_acc      <- as.numeric(fold_cm$overall["Accuracy"])
  
  cat(sprintf("  Fold %d/%d: Validation Accuracy = %.4f (%.2f%%) [Size: %d rows]\n",
              k, k_folds, fold_acc, fold_acc * 100, length(test_idx)))
  
  fold_metrics[[k]] <- fold_acc
}
cat("============================================================\n")
cat(sprintf("   5-FOLD CV COMPLETE. Mean Fold Accuracy = %.4f (+/- %.4f)\n",
            mean(unlist(fold_metrics)), sd(unlist(fold_metrics))))
cat("============================================================\n\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Overall Out-Of-Fold Benchmark & Report Metrics (Accuracy, Precision, Recall, ROC-AUC, F1-Score, Confusion Matrix)
# ---------------------------------------------------------
oof_pred_idx <- apply(oof_prob, 1, which.max)
oof_pred_val <- colnames(oof_prob)[oof_pred_idx]
oof_pred_fac <- factor(oof_pred_val, levels = c("1", "2", "3", "4", "5"))
act_fac      <- factor(df_full$target_col, levels = c("1", "2", "3", "4", "5"))
cm_oof <- confusionMatrix(oof_pred_fac, act_fac)
acc_oof <- as.numeric(cm_oof$overall["Accuracy"])
prec_by_class <- as.numeric(cm_oof$byClass[, "Pos Pred Value"])
rec_by_class  <- as.numeric(cm_oof$byClass[, "Sensitivity"])
prec_by_class[is.na(prec_by_class)] <- 0
rec_by_class[is.na(rec_by_class)]   <- 0
f1_by_class <- ifelse((prec_by_class + rec_by_class) > 0, 
                      2 * (prec_by_class * rec_by_class) / (prec_by_class + rec_by_class), 0)
roc_auc_by_class <- sapply(1:5, function(i) {
  cls_name <- levels(act_fac)[i]
  act_bin  <- ifelse(act_fac == cls_name, 1, 0)
  r_obj    <- tryCatch(pROC::roc(act_bin, oof_prob[, i]), error = function(e) NULL)
  if (!is.null(r_obj)) as.numeric(r_obj$auc) else NA
})
actual_counts <- as.numeric(table(act_fac))
pred_counts   <- as.numeric(table(oof_pred_fac))
diff_vec      <- pred_counts - actual_counts
diff_str      <- ifelse(diff_vec >= 0, paste0("+", diff_vec), as.character(diff_vec))
report_df <- data.frame(
  Class        = levels(act_fac),
  Actual_Count = actual_counts,
  Pred_Count   = pred_counts,
  Diff         = diff_str,
  Precision    = round(prec_by_class, 4),
  Recall       = round(rec_by_class, 4),
  F1_Score     = round(f1_by_class, 4),
  ROC_AUC      = round(roc_auc_by_class, 4)
)
macro_prec    <- mean(prec_by_class)
macro_rec     <- mean(rec_by_class)
macro_f1      <- mean(f1_by_class)
macro_roc_auc <- mean(roc_auc_by_class, na.rm = TRUE)
cat(sprintf("============================================================\n"))
cat(sprintf("   5-CLASS RANDOM FOREST - 5-FOLD CV OUT-OF-FOLD BENCHMARK\n"))
cat(sprintf("============================================================\n"))
cat(sprintf("  Overall OOF Accuracy : %.4f (%.2f%%)\n", acc_oof, acc_oof * 100))
cat(sprintf("  Macro Precision      : %.4f\n", macro_prec))
cat(sprintf("  Macro Recall (Sens)  : %.4f\n", macro_rec))
cat(sprintf("  Macro F1-Score       : %.4f\n", macro_f1))
cat(sprintf("  Macro ROC-AUC        : %.4f\n", macro_roc_auc))
cat(sprintf("============================================================\n\n"))
cat("Per-Class OOF Performance & Count Summary:\n")
print(report_df)
cat("\nAggregated 5-Fold Confusion Matrix (Rows: Predicted, Columns: Actual):\n")
print(cm_oof$table)
cat(sprintf("============================================================\n\n"))
# Export CSV Report
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
write.csv(report_df, file = file.path(reports_dir, "rf_all_5fold_cv_report.csv"), row.names = FALSE)
cat("5-Fold CV Out-Of-Fold CSV Report written to: reports/rf_all_5fold_cv_report.csv\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Diagnostic Plots (Metrics Bar Chart)
# ---------------------------------------------------------
plots_dir <- "../plots"
if (!dir.exists(plots_dir)) plots_dir <- "plots"
if (!dir.exists(plots_dir)) dir.create(plots_dir, recursive = TRUE)
metrics_summary <- data.frame(
  Metric = factor(c("Accuracy", "Macro_Precision", "Macro_Recall", "Macro_F1_Score", "Macro_ROC_AUC"),
                  levels = c("Accuracy", "Macro_Precision", "Macro_Recall", "Macro_F1_Score", "Macro_ROC_AUC")),
  Score  = c(acc_oof, macro_prec, macro_rec, macro_f1, macro_roc_auc)
)
p_bar <- ggplot(metrics_summary, aes(x = Metric, y = Score, fill = Metric)) +
  geom_bar(stat = "identity", width = 0.5) +
  geom_text(aes(label = sprintf("%.3f", Score)), vjust = -0.3, size = 4, fontface = "bold") +
  theme_minimal() +
  scale_fill_brewer(palette = "Set2") +
  labs(title = "5-Fold Cross-Validation Metrics (Random Forest)",
       subtitle = "Out-Of-Fold Performance Across 29 Predictor Features",
       y = "Metric Score", x = "") +
  theme(plot.title = element_text(face = "bold", size = 13, hjust = 0.5),
        plot.subtitle = element_text(size = 10, hjust = 0.5),
        legend.position = "none")
ggsave(file.path(plots_dir, "rf_all_5fold_cv_barchart.png"), plot = p_bar, width = 9, height = 5, dpi = 300)
cat("5-Fold CV Metrics Bar Chart saved to: plots/rf_all_5fold_cv_barchart.png\n")
p_bar

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Train & Save Final Production Random Forest Model
# ---------------------------------------------------------
set.seed(config$training$random_state)
cat("Training final production Random Forest model on complete dataset...\n")
preproc_final <- preProcess(df_full[, cont_cols, drop = FALSE], method = c("center", "scale"))
df_full_scaled <- predict(preproc_final, df_full)
if (is_ranger) {
  final_rf_model <- ranger::ranger(
    formula      = target_col ~ .,
    data         = df_full_scaled,
    num.trees    = 100,
    probability  = TRUE,
    seed         = config$training$random_state,
    verbose      = FALSE
  )
} else {
  final_rf_model <- randomForest::randomForest(
    target_col ~ .,
    data  = df_full_scaled,
    ntree = 100
  )
}
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)
model_path <- file.path(deploy_dir, "rf_all_model.rds")
saveRDS(list(model = final_rf_model, preproc = preproc_final, is_ranger = is_ranger), file = model_path)
cat("Final 5-Fold Trained Multi-Class Random Forest Model saved to:", model_path, "\n")